# Lab 1 - Agentic AI Chain: Business Opportunity Finder

**Contributor:** Akhil Guska ([@akhilaguska27](https://github.com/akhilaguska27))

This notebook solves the Lab 1 exercise using a 3-step LLM chain:
1. Ask the LLM to pick a business area ripe for Agentic AI
2. Ask the LLM to identify a pain point in that industry
3. Ask the LLM to propose an Agentic AI solution for that pain point

The key pattern: the output of each call becomes the input of the next.
This chaining is what makes it 'agentic' - the model builds on its own reasoning across steps.

Two versions are included: one using OpenAI and one using Anthropic (Claude).

Common errors and fixes are documented at the bottom of this notebook.

## Setup

In [ ]:
# Uncomment to install if needed
# !pip install openai anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Load keys from .env file in your project root
# .env should contain:
#   OPENAI_API_KEY=sk-proj-...
#   ANTHROPIC_API_KEY=sk-ant-...   (optional for this lab)
load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

print(f"OpenAI key found: {bool(openai_key)}")
print(f"Anthropic key found: {bool(anthropic_key)} (optional)")

## Solution A - OpenAI (GPT-4.1-mini)

In [ ]:
from openai import OpenAI

openai_client = OpenAI()
GPT_MODEL = "gpt-4.1-mini"

def call_gpt(prompt):
    """Send a single prompt to GPT and return the response text."""
    response = openai_client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
# Call 1 - Pick a business area
prompt_1 = (
    "You are an expert in AI and business strategy. "
    "Pick ONE business area or industry worth exploring for an Agentic AI opportunity. "
    "Name the area and give a 2-3 sentence explanation of why it is a strong candidate."
)

business_area = call_gpt(prompt_1)
display(Markdown("**Call 1 - Business Area:**"))
display(Markdown(business_area))

In [ ]:
# Call 2 - Find a pain point in that area
# business_area from Call 1 is injected directly into this prompt
prompt_2 = (
    f"Here is a business area identified for Agentic AI exploration:\n\n{business_area}\n\n"
    "Identify ONE specific pain point in this industry that is repetitive, time-consuming, "
    "or error-prone for humans and would be a strong candidate for an Agentic AI solution. "
    "Describe it in 3-4 sentences."
)

pain_point = call_gpt(prompt_2)
display(Markdown("**Call 2 - Pain Point:**"))
display(Markdown(pain_point))

In [ ]:
# Call 3 - Propose an Agentic AI solution
# pain_point from Call 2 is injected here
prompt_3 = (
    f"Here is a pain point in a specific industry:\n\n{pain_point}\n\n"
    "Propose a detailed Agentic AI solution. Include:\n"
    "1. What the agent does\n"
    "2. How it works step by step\n"
    "3. What tools or data it needs\n"
    "4. Why this is better than a non-agentic approach\n"
    "5. The expected business impact"
)

solution = call_gpt(prompt_3)
display(Markdown("**Call 3 - Agentic AI Solution:**"))
display(Markdown(solution))

In [ ]:
# Full chain summary
display(Markdown("---\n**Full Chain Summary (OpenAI)**"))
display(Markdown(f"**Business Area:** {business_area}"))
display(Markdown(f"**Pain Point:** {pain_point}"))
display(Markdown(f"**Solution:** {solution}"))

## Solution B - Anthropic (Claude)

The logic is identical to Solution A. The differences are SDK syntax only:
- Use `.messages.create()` instead of `.chat.completions.create()`
- `max_tokens` is required by Anthropic (OpenAI does not require it)
- Response text is at `response.content[0].text` not `response.choices[0].message.content`

Note: ANTHROPIC_API_KEY must be set in your .env file to run this section.

In [ ]:
from anthropic import Anthropic

# Anthropic() auto-reads ANTHROPIC_API_KEY from environment
# If you get TypeError: Could not resolve authentication method
# the key is missing - see the error guide at the bottom of this notebook
claude_client = Anthropic()
CLAUDE_MODEL = "claude-sonnet-4-5"

def call_claude(prompt):
    """Send a single prompt to Claude and return the response text.
    
    Unlike OpenAI, Anthropic requires max_tokens to be set explicitly.
    Response text is at response.content[0].text
    """
    response = claude_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

In [ ]:
# Call 1 - same prompt as OpenAI version
business_area_claude = call_claude(prompt_1)
display(Markdown("**Call 1 - Business Area (Claude):**"))
display(Markdown(business_area_claude))

In [ ]:
# Call 2 - inject Claude's Call 1 output
prompt_2_claude = (
    f"Here is a business area identified for Agentic AI exploration:\n\n{business_area_claude}\n\n"
    "Identify ONE specific pain point in this industry that is repetitive, time-consuming, "
    "or error-prone for humans and would be a strong candidate for an Agentic AI solution. "
    "Describe it in 3-4 sentences."
)

pain_point_claude = call_claude(prompt_2_claude)
display(Markdown("**Call 2 - Pain Point (Claude):**"))
display(Markdown(pain_point_claude))

In [ ]:
# Call 3 - inject Claude's Call 2 output
prompt_3_claude = (
    f"Here is a pain point in a specific industry:\n\n{pain_point_claude}\n\n"
    "Propose a detailed Agentic AI solution. Include:\n"
    "1. What the agent does\n"
    "2. How it works step by step\n"
    "3. What tools or data it needs\n"
    "4. Why this is better than a non-agentic approach\n"
    "5. The expected business impact"
)

solution_claude = call_claude(prompt_3_claude)
display(Markdown("**Call 3 - Agentic AI Solution (Claude):**"))
display(Markdown(solution_claude))

## GPT vs Claude Comparison

Both models follow the same 3-step chain. Each run may produce different industries and solutions depending on what the model picks in Call 1 - that is expected behaviour.

In [ ]:
display(Markdown("**GPT business area:** " + business_area[:300] + "..."))
display(Markdown("**Claude business area:** " + business_area_claude[:300] + "..."))

---

## Common Errors and Fixes

These are the errors students most frequently hit in this lab.

---

### TypeError: Could not resolve authentication method

Your Anthropic API key is missing or not loaded. The key is optional for Lab 1, but if you try to run Solution B without it you will hit this error.

Fix 1 - add to your .env file (recommended):
```
ANTHROPIC_API_KEY=sk-ant-your-key-here
```

Fix 2 - pass the key directly in code (quick testing only, do not commit this):
```python
claude_client = Anthropic(api_key="sk-ant-your-key-here")
```

Fix 3 - set it in terminal before launching Jupyter:
```bash
export ANTHROPIC_API_KEY="sk-ant-your-key-here"
```

Get your key at: https://console.anthropic.com

---

### TypeError: Missing required argument: max_tokens

Anthropic requires `max_tokens`, OpenAI does not. Always include it with the Anthropic SDK:

```python
# Wrong
response = claude_client.messages.create(
    model="claude-sonnet-4-5",
    messages=messages
)

# Correct
response = claude_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    messages=messages
)
```

---

### AttributeError on response object when using Claude

OpenAI and Anthropic return different response structures:

| | OpenAI | Anthropic |
|---|---|---|
| Get response text | `response.choices[0].message.content` | `response.content[0].text` |
| Method | `.chat.completions.create()` | `.messages.create()` |
| max_tokens | Optional | Required |

---

### AuthenticationError: invalid_api_key

Your key is wrong, expired, or has no credits.

- OpenAI keys start with `sk-proj-` (newer) or `sk-` (older)
- Anthropic keys start with `sk-ant-`
- No quotes or spaces around the key in your .env file
- Check credits: platform.openai.com/usage or console.anthropic.com

Correct .env format:
```
OPENAI_API_KEY=sk-proj-abc123
ANTHROPIC_API_KEY=sk-ant-abc123
```

---

### .env keys not loading

The .env file must be in the same directory as your notebook. Check with:

```python
import os
print(os.getcwd())  # your .env must be in this folder
```

Always use `override=True` so the key refreshes on re-run:
```python
load_dotenv(override=True)
```